# D-optimal trial design — can a better-designed trial rescue the flat parameter?

**NOT FOR CLINICAL USE.** Design/population-level only; illustrative, unverified parameters. Executed in CI (nbmake).

`onkos.identify` (v0.22) evaluates whether a *given* sampling schedule can estimate a model's parameters, from the design Fisher information `M = SᵀS`. But a pharmacometrician *chooses* the sampling times. `onkos.design` adds that choice: the **D-optimal schedule** of `N` measurements (the times maximizing `det M` — the smallest joint confidence ellipsoid), scored against a uniform schedule of the same budget.

Because the information is **additive over timepoints** (`M = Σᵢ sᵢsᵢᵀ`), the sensitivities are computed once on a dense grid and the optimization is pure linear algebra. The payload: the optimal design separates the *circumstantially* flat parameter (rescued by a better trial) from the *structurally* flat one (unidentifiable under any schedule of this budget). This is `docs/specs/research/optimal-design.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.design import optimal_schedule, d_optimal_rows

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The pure selection core

`d_optimal_rows` greedily picks the rows (timepoints) that most increase `log det M`. On two large orthogonal rows plus two tiny ones, the D-optimal 2-subset is the two large orthogonal rows.

In [ ]:
ss = np.array([[10., 0.], [0., 10.], [1., 0.], [0., 1.]])
print('D-optimal 2-subset:', sorted(d_optimal_rows(ss, 2)), '(the two large orthogonal rows)')

## Claret NSCLC: design helps, but cannot rescue the deeply flat kL

The D-optimal schedule concentrates samples at the kill phase and regrowth onset, tightening every parameter (D-efficiency > 1). The borderline resistance term λ is rescued across the 50% identifiability line; the deeply flat growth rate kL is not.

In [ ]:
od = optimal_schedule(ds, 'resistance.claret_2009.tgi', context=ctx, n_samples=7, horizon=48.0)
print(f"{'param':<8}{'uniform RSE':>13}{'D-opt RSE':>12}{'identifiable (D-opt)':>22}")
for s in od.uniform.rse_percent:
    print(f"{s:<8}{od.uniform.rse_percent[s]:>12.0f}%{od.optimal.rse_percent[s]:>11.0f}%"
          f"{('yes' if s in od.optimal.identifiable else 'NO'):>22}")
print(f"\nD-optimal schedule (wk) = {[round(t) for t in od.optimal.schedule]}")
print(f'D-efficiency over uniform = {od.d_efficiency:.2f}x')
print(f'rescued by the better design = {sorted(set(od.optimal.identifiable) - set(od.uniform.identifiable))}')
print(f'structurally flat (best design still fails) = {od.structurally_flat}')

## The control: a model the design *can* serve

The 2-parameter Wang biexponential is fully identifiable under the optimal design — proving the kL failure above is the *model's* structure, not the optimizer. Optimal design is the control that separates a design problem from a model problem.

In [ ]:
for rid in ['tgi_metrics.wang_2009.biexponential', 'resistance.claret_2009.tgi']:
    o = optimal_schedule(ds, rid, context=ctx, n_samples=7, horizon=48.0)
    verdict = 'all identifiable' if not o.structurally_flat else f'flat: {o.structurally_flat}'
    print(f"{rid.split('.')[-1]:<16} {len(o.uniform.rse_percent)} params, D-eff {o.d_efficiency:.2f}x -> {verdict}")

## Guardrails

A design analysis carries the record's propagated tier unchanged, is design/population-level only (never a per-patient schedule or a dosing/therapy choice), and is honest about the greedy heuristic (guaranteed only ≥ uniform). A non-trajectory (survival/transform) record is rejected.

In [ ]:
print('tier (unchanged) =', od.tier, '| clinical use =', od.to_dict()['onkos:clinicalUse'][:24], '...')
try:
    optimal_schedule(ds, 'survival_link.nsclc_os_week8', context=ctx)
except ValueError as e:
    print('survival link rejected:', str(e)[:60], '...')

## The takeaway

Optimal design is what separates *circumstantial* from *structural* unidentifiability. The best schedule a fixed budget allows rescues the parameter a better trial could pin down (λ here) and leaves the one no trial of this budget can (kL) — so kL's huge reported CV is a flat-likelihood artifact, not biological spread. This is the rigorous capstone to the v0.22 identifiability work; CI enforces it (`tests/test_design.py`).